## 1 · HF chaos

In [ ]:
!pip install hf_xet

In [ ]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
# Force ALL HuggingFace caches to D: — must run BEFORE any HF import
os.environ["HF_HOME"]           = r"/mnt/d/hf_cache"
os.environ["HF_HUB_CACHE"]      = r"/mnt/d/hf_cache\hub"
os.environ["HF_DATASETS_CACHE"] = r"/mnt/d/hf_cache\datasets"
os.environ["HF_XET_CACHE"]      = r"/mnt/d/hf_cache\xet"

# Sanity check
print("HF_HOME:", os.environ["HF_HOME"])

## 2 · Imports

In [ ]:
from __future__ import annotations

import math
from typing import Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

from transformers import (
    Dinov2Config,
    Dinov2Model,
    Dinov2ForImageClassification,
    AutoImageProcessor,
)
from transformers.models.dinov2.modeling_dinov2 import (
    Dinov2Layer,
    Dinov2SelfOutput,
)
from PIL import Image
import requests

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

import os

In [ ]:
!nvidia-smi

In [ ]:
class GatedDinov2SelfAttention(nn.Module):
    def __init__(self, config: Dinov2Config) -> None:
        super().__init__()

        if config.hidden_size % config.num_attention_heads != 0:
            raise ValueError(
                f"hidden_size ({config.hidden_size}) is not divisible by "
                f"num_attention_heads ({config.num_attention_heads})"
            )

        self.num_attention_heads = config.num_attention_heads
        self.attention_head_size = config.hidden_size // config.num_attention_heads
        self.all_head_size = self.num_attention_heads * self.attention_head_size

        # Standard QKV projections
        self.query = nn.Linear(config.hidden_size, self.all_head_size, bias=config.qkv_bias)
        self.key = nn.Linear(config.hidden_size, self.all_head_size, bias=config.qkv_bias)
        self.value = nn.Linear(config.hidden_size, self.all_head_size, bias=config.qkv_bias)
        self.dropout = nn.Dropout(config.attention_probs_dropout_prob)
        self.gate_proj = nn.Linear(config.hidden_size, self.all_head_size)

        nn.init.zeros_(self.gate_proj.weight)
        nn.init.constant_(self.gate_proj.bias, 2.0)

    def transpose_for_scores(self, x: torch.Tensor) -> torch.Tensor:
        """Reshape (B, N, H*D) -> (B, H, N, D)."""
        new_shape = x.size()[:-1] + (self.num_attention_heads, self.attention_head_size)
        x = x.view(new_shape)
        return x.permute(0, 2, 1, 3)

    def forward(
        self,
        hidden_states: torch.Tensor,
        head_mask: Optional[torch.Tensor] = None,
        output_attentions: bool = False,
        **kwargs,
    ) -> Tuple[torch.Tensor, ...]:
        batch_size, seq_len, _ = hidden_states.shape

        gate_score = self.gate_proj(hidden_states)
        gate_score = self.transpose_for_scores(gate_score)  # (B, H, N, D)
        gate_score = torch.sigmoid(gate_score)

        query_layer = self.transpose_for_scores(self.query(hidden_states))
        key_layer = self.transpose_for_scores(self.key(hidden_states))
        value_layer = self.transpose_for_scores(self.value(hidden_states))

        attention_scores = torch.matmul(query_layer, key_layer.transpose(-1, -2))
        attention_scores = attention_scores / math.sqrt(self.attention_head_size)

        attention_probs = F.softmax(attention_scores, dim=-1)
        attention_probs = self.dropout(attention_probs)

        if head_mask is not None:
            attention_probs = attention_probs * head_mask

        context_layer = torch.matmul(attention_probs, value_layer)  # (B, H, N, D)

        context_layer = context_layer * gate_score

        # (B, H, N, D) -> (B, N, H*D)
        context_layer = context_layer.permute(0, 2, 1, 3).contiguous()
        context_layer = context_layer.view(batch_size, seq_len, self.all_head_size)

        outputs = (context_layer, attention_probs) if output_attentions else (context_layer, None)
        return outputs

In [ ]:
class GatedDinov2Attention(nn.Module):
    """Dinov2Attention with gated self-attention"""
    def __init__(self, config: Dinov2Config) -> None:
        super().__init__()
        self.attention = GatedDinov2SelfAttention(config)
        self.output = Dinov2SelfOutput(config)

    def forward(
        self,
        hidden_states: torch.Tensor,
        head_mask: Optional[torch.Tensor] = None,
        output_attentions: bool = False,
        **kwargs,
    ) -> Tuple[torch.Tensor, ...]:
        self_attn_output, attn_weights = self.attention(
            hidden_states,
            head_mask=head_mask,
            output_attentions=output_attentions,
            **kwargs,
        )
        attention_output = self.output(self_attn_output, hidden_states)

        outputs = (attention_output,)
        if output_attentions:
            outputs = outputs + (attn_weights,)
        return outputs


def _replace_attention_in_layer(layer: Dinov2Layer, config: Dinov2Config) -> None:
    layer.attention = GatedDinov2Attention(config).to(device)


def inject_gated_attention(model: nn.Module, config: Dinov2Config) -> nn.Module:
    # replace every Dinov2Attention
    encoder = None
    if hasattr(model, "encoder"):
        encoder = model.encoder
    elif hasattr(model, "dinov2") and hasattr(model.dinov2, "encoder"):
        encoder = model.dinov2.encoder

    if encoder is None:
        raise ValueError("Could not locate the encoder inside the model.")

    for layer in encoder.layer:
        _replace_attention_in_layer(layer, config)

    return model

In [ ]:
class GatedDinov2Model(Dinov2Model):
    # model = GatedDinov2Model.from_pretrained("facebook/dinov2-base")
    def __init__(self, config: Dinov2Config) -> None:
        super().__init__(config)
        inject_gated_attention(self, config)
        self.post_init()


class GatedDinov2ForImageClassification(Dinov2ForImageClassification):
    def __init__(self, config: Dinov2Config) -> None:
        super().__init__(config)
        inject_gated_attention(self, config)
        self.post_init()

In [ ]:
HF_TOKEN = "token"
from huggingface_hub import login
login(token=HF_TOKEN)

import os
print("HF_HOME:", os.environ.get("HF_HOME"))
print("CWD:", os.getcwd())


In [ ]:
from datasets import load_dataset
from datasets import Image as HFImage
from huggingface_hub import snapshot_download
import time

for attempt in range(10):
    try:
        snapshot_download(
            repo_id="ILSVRC/imagenet-1k",
            repo_type="dataset",
            token=HF_TOKEN,
            max_workers=4,
        )
        break
    except Exception as e:
        print(f"Attempt {attempt+1} failed: {type(e).__name__}: {e}")
        time.sleep(10)
        
ds = load_dataset("ILSVRC/imagenet-1k", token=HF_TOKEN)
train_ds = ds["train"]
val_ds = ds["validation"]
train_ds = train_ds.cast_column("image", HFImage(decode=False))
val_ds = val_ds.cast_column("image", HFImage(decode=False))

NUM_TRAIN = len(train_ds)
NUM_VAL = len(val_ds)
NUM_CLASSES = 1000
NUM_TRAIN = len(train_ds)
NUM_VAL = len(val_ds)
NUM_CLASSES = 1000

print(f"Train: {NUM_TRAIN} | Val: {NUM_VAL} | Classes: {NUM_CLASSES}")

In [ ]:
# DINOv2-Small / ViT-S/14 
config = Dinov2Config(
    hidden_size=384,
    num_hidden_layers=12,
    num_attention_heads=6,
    intermediate_size=1536,
    image_size=224,
    patch_size=14,
    num_labels=1000, # ImageNet-1K
)

cls_model = Dinov2ForImageClassification(config)
#inject_gated_attention(cls_model, config)
cls_model = cls_model.to(device)

for p in cls_model.parameters():
    p.requires_grad = True

total_params = sum(p.numel() for p in cls_model.parameters())
trainable = sum(p.numel() for p in cls_model.parameters() if p.requires_grad)
print(f"Total params: {total_params}")
print(f"Trainable params: {trainable}")


In [ ]:
import torch, torchvision
print(torch.__version__, torchvision.__version__)

In [ ]:
import torch
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import LinearLR
from torchvision import transforms
from PIL import Image, ImageFile
import tqdm
import io

# Reject images bigger than 200MP to prevent OOM from malicious metadata
Image.MAX_IMAGE_PIXELS = 200000000
ImageFile.LOAD_TRUNCATED_IMAGES = True

EPOCHS = 20
BATCH_SIZE = 192 # 28gb of vram
GRAD_ACCUM = 2
LR = 2e-4
WEIGHT_DECAY = 0.01
NUM_WORKERS = 4                   
SAVE_EVERY_N_STEPS = 500          
BEST_CKPT = "gated_dinov2_scratch_best.pt"
LAST_CKPT = "gated_dinov2_scratch_last.pt"

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(), # was overfitting
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# utf8 error needs to load image from bytes, imagenet is lowkey so annoying 
def train_collate(batch):
    images, labels = [], []
    for x in batch:
        try:
            img_bytes = x["image"]["bytes"]
            if img_bytes is None:
                continue
            with Image.open(io.BytesIO(img_bytes)) as pil_img:
                img = pil_img.convert("RGB")
            images.append(train_transform(img))
            labels.append(x["label"])
        except (UnicodeDecodeError, OSError, SyntaxError, ValueError, KeyError):
            continue
    if not images:
        return torch.empty(0, 3, 224, 224), torch.empty(0, dtype=torch.long)
    return torch.stack(images), torch.tensor(labels, dtype= torch.long)

def val_collate(batch):
    images, labels = [], []
    for x in batch:
        try:
            img_bytes = x["image"]["bytes"]
            if img_bytes is None:
                continue
            with Image.open(io.BytesIO(img_bytes)) as pil_img:
                img = pil_img.convert("RGB")
            images.append(val_transform(img))
            labels.append(x["label"])
        except (UnicodeDecodeError, OSError, SyntaxError, ValueError, KeyError):
            continue
    if not images:
        return torch.empty(0, 3, 224, 224), torch.empty(0, dtype=torch.long)
    return torch.stack(images), torch.tensor(labels, dtype=torch.long)

train_loader = DataLoader(train_ds, collate_fn=train_collate, shuffle= True, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory= True)
val_loader = DataLoader(val_ds, collate_fn=val_collate, shuffle=False, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=True)
optimizer = torch.optim.AdamW(cls_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY) 
criterion = torch.nn.CrossEntropyLoss()
scheduler = LinearLR(optimizer, start_factor= 1.0, end_factor=0.0, total_iters=EPOCHS * (len(train_ds) //(BATCH_SIZE * GRAD_ACCUM))) # linear weight decay

train_losses, train_accs, val_accs = [], [], []
best_val_acc = 0.0

# Resume if last checkpoint exist
start_epoch = 0
import os
if os.path.exists(LAST_CKPT):
    ckpt = torch.load(LAST_CKPT, map_location=device)
    cls_model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    start_epoch = ckpt.get("epoch", 0)
    best_val_acc = ckpt.get("best_val_acc", 0.0)
    train_losses = ckpt.get("train_losses", [])
    train_accs = ckpt.get("train_accs", [])
    val_accs = ckpt.get("val_accs", [])

for epoch in range(start_epoch, EPOCHS):
    cls_model.train()
    running_loss, correct, seen = 0.0, 0, 0
    optimizer.zero_grad()
    pbar = tqdm.tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")
    for i, (images, labels) in enumerate(pbar):
        if images.numel() == 0:
            continue
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits = cls_model(images).logits
        loss = criterion(logits, labels) / GRAD_ACCUM
        loss.backward()

        if (i + 1) % GRAD_ACCUM == 0:
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        running_loss += loss.item() * images.size(0) * GRAD_ACCUM
        correct += (logits.argmax(1) == labels).sum().item()
        seen += images.size(0)
        pbar.set_postfix(loss=f"{running_loss/seen}", acc=f"{correct/seen}", lr=f"{optimizer.param_groups[0]['lr']}") # lr wasn't devreaseing before

        if (i + 1) % SAVE_EVERY_N_STEPS == 0:
            torch.save({
                "epoch": epoch,
                "model_state_dict": cls_model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "best_val_acc": best_val_acc,
                "train_losses": train_losses, "train_accs": train_accs, "val_accs": val_accs,
            }, LAST_CKPT)

    train_loss = running_loss / seen
    train_acc  = correct / seen

    # eval loop
    cls_model.eval()
    v_correct, v_seen = 0, 0
    with torch.no_grad():
        for images, labels in tqdm.tqdm(val_loader, desc="Val", leave=False):
            if images.numel() == 0:
                continue
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            logits = cls_model(images).logits
            v_correct += (logits.argmax(1) == labels).sum().item()
            v_seen += images.size(0)
    val_acc = v_correct / v_seen

    train_losses.append(train_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    # resume from ech epoch
    torch.save({
        "epoch": epoch + 1,
        "model_state_dict": cls_model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_val_acc": max(best_val_acc, val_acc),
        "train_losses": train_losses, "train_accs": train_accs, "val_accs": val_accs,
    }, LAST_CKPT)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({"epoch": epoch + 1, "model_state_dict": cls_model.state_dict(), "val_acc": val_acc, "config": config.to_dict()}, BEST_CKPT)

In [ ]:
import json

with open("metrics_base.json", "w") as f:
    json.dump({
        "train_losses": train_losses,
        "train_accs": train_accs,
        "val_accs": val_accs,
        "best_val_acc": best_val_acc,
    }, f, indent=2)

In [ ]:
import json
import matplotlib.pyplot as plt

with open("metrics_base.json") as f:
    m = json.load(f)

epochs = list(range(len(m["train_losses"])))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

# Loss
ax1.plot(epochs, m["train_losses"], marker="o", linewidth=2, color="#1f77b4")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Train loss")
ax1.set_title("Training loss")
ax1.grid(True, alpha=0.3)

# Accuracy
ax2.plot(epochs, m["train_accs"], marker="o", linewidth=2, label="Train", color="#1f77b4")
ax2.plot(epochs, m["val_accs"],   marker="s", linewidth=2, label="Val",   color="#ff7f0e")
ax2.axhline(m["best_val_acc"], color="gray", linestyle="--", alpha=0.5,
            label=f"Best val: {m['best_val_acc']:.4f}")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Top-1 accuracy")
ax2.set_title("Accuracy")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("metrics_base_plots.png", dpi=150, bbox_inches="tight")
plt.show()